# 83 — Luật chơi Việt Nam

backtrader được viết cho thị trường Mỹ: phí một tỷ lệ cho cả hai chiều, mua lẻ
từng cổ phiếu, giá đi đâu cũng khớp, bán ngay hôm sau, bán khống thoải mái. Sàn
Việt Nam khác ở cả năm chỗ. Notebook này dựng tay từng luật, đo nó làm kết quả
đổi bao nhiêu, rồi đối chiếu với bản đã gom vào `bt_vn`.

| Luật | backtrader mặc định | Việt Nam |
|---|---|---|
| Phí, thuế | một tỷ lệ, hai chiều như nhau | phí môi giới hai chiều + **thuế 0,1% chỉ chiều bán** |
| Khối lượng | cổ phiếu lẻ | bội số **lô 100** |
| Giá | khớp ở mọi mức | trong biên độ **±7% HOSE / ±10% HNX / ±15% UPCOM**, theo bước giá |
| Trần/sàn trắng bên | vẫn khớp | **không khớp** được |
| Thanh toán | bán được ngay phiên sau | hàng về **T+2** |
| Bán khống | được | **không** với cổ phiếu |

Cạm bẫy được đo: **biên độ phải tính trên giá thô**, không phải giá điều chỉnh.

In [1]:
import sys
from collections import deque
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import math

import backtrader as bt
import numpy as np
import pandas as pd

import finlens
from finlens_examples import ap_dung_theme, bar_ngang, hom_nay, lui_ngay
from finlens_examples import bt_vn

ap_dung_theme()
client = finlens.client()
HOM_NAY = hom_nay(client)

D:\finlens\finlens-python\finlens-python-example\venv\Lib\site-packages\anyio\_core\_tasks.py:275: UpgradeAvailableWarning: finlens 1.5.0 đã có (đang dùng 1.4.0). Nâng cấp: pip install -U finlens
  retval = await self._coro


## 1 · Dữ liệu

Mười mã HOSE biến động mạnh, thanh khoản tốt — loại cổ phiếu hay chạm trần sàn,
nên luật biên độ có việc để làm. Tải **hai lần**: giá điều chỉnh để tính lợi
suất, giá thô (`adjusted=False`) để kiểm biên độ.

In [2]:
RO = ["CII", "DIG", "DXG", "GEX", "HHS", "NVL", "PDR", "TCH", "VIX", "VND"]
SO_NAM = 5

gia = bt_vn.bo_phien_chua_dong(client.eod.stock.ohlcv(RO, start=lui_ngay(HOM_NAY, nam=SO_NAM)), client)
gia_tho = bt_vn.bo_phien_chua_dong(
    client.eod.stock.ohlcv(RO, start=lui_ngay(HOM_NAY, nam=SO_NAM), adjusted=False), client
)
SAN = client.meta.symbols(exchange="HOSE", kind="stock").set_index("symbol").loc[RO, "exchange"].to_dict()

print(f"Điều chỉnh: {len(gia):,} dòng · price_basis = {gia.attrs['finlens']['price_basis']}")
print(f"Thô:        {len(gia_tho):,} dòng · price_basis = {gia_tho.attrs['finlens']['price_basis']}")
print(f"Sàn: {sorted(set(SAN.values()))} · {gia['date'].min():%d/%m/%Y} → {gia['date'].max():%d/%m/%Y}")

Điều chỉnh: 12,450 dòng · price_basis = adjusted
Thô:        12,450 dòng · price_basis = raw
Sàn: ['HOSE'] · 20/10/2021 → 19/10/2026


## 2 · Phí và thuế: `CommInfoBase` riêng

`cerebro.broker.setcommission(commission=0.0015)` tính 0,15% cho **cả hai
chiều**. Ở Việt Nam chiều bán còn chịu thuế thu nhập cá nhân 0,1% trên giá trị
bán — dù lãi hay lỗ. Một vòng mua-bán vì thế tốn `2 × 0,15% + 0,1% = 0,4%`,
không phải 0,3%.

backtrader tính phí qua `_getcommission(size, price, pseudoexec)`; `size` âm là
lệnh bán. Ghi đè đúng hàm đó:

In [3]:
class PhiTay(bt.CommInfoBase):
    """Phí môi giới hai chiều + thuế bán, tính trên giá trị khớp (VND)."""

    params = (
        ("phi", 0.0015),
        ("thue_ban", 0.001),
        ("stocklike", True),  # cổ phiếu: trả đủ tiền, không ký quỹ
        ("commtype", bt.CommInfoBase.COMM_PERC),
        ("percabs", True),  # 0.0015 là 0,15% — không phải 0.0015%
    )

    def _getcommission(self, size, price, pseudoexec):
        gia_tri = abs(size) * price
        return gia_tri * (self.p.phi + (self.p.thue_ban if size < 0 else 0.0))


mac_dinh = bt.CommissionInfo(commission=0.0015, percabs=True)  # thứ setcommission() tạo ra
tay = PhiTay()
bang_phi = pd.DataFrame(
    {
        "mặc định (VND)": [mac_dinh.getcommission(1_000, 20_000), mac_dinh.getcommission(-1_000, 21_000)],
        "luật VN (VND)": [tay.getcommission(1_000, 20_000), tay.getcommission(-1_000, 21_000)],
    },
    index=["mua 1.000 cp × 20.000", "bán 1.000 cp × 21.000"],
)
bang_phi.loc["cả vòng"] = bang_phi.sum()
bang_phi.style.format("{:,.0f}")

,mặc định (VND),luật VN (VND)
mua 1.000 cp × 20.000,"30,000","30,000"
bán 1.000 cp × 21.000,"31,500","52,500"
cả vòng,"61,500","82,500"


Chiều bán tốn thêm đúng 21.000 đồng — 0,1% của 21 triệu. Nhỏ ở một lệnh, nhưng
chiến lược quay vòng 30 lần một năm thì đó là 3% vốn mỗi năm chỉ riêng tiền thuế.

## 3 · Lô 100: `Sizer` riêng

Sàn chỉ nhận lệnh thường theo bội số 100 cổ phiếu. Sizer quyết định khối lượng
mỗi khi chiến lược gọi `self.buy()` không kèm `size`:

In [4]:
class LoTay(bt.Sizer):
    """Mua bằng ``ty_le`` tiền mặt, làm tròn XUỐNG lô 100; bán thì bán hết vị thế."""

    params = (("ty_le", 0.95), ("lo", 100))

    def _getsizing(self, comminfo, cash, data, isbuy):
        if not isbuy:
            return self.broker.getposition(data).size
        return int(cash * self.p.ty_le / data.close[0] // self.p.lo) * self.p.lo


for tien, gia_cp in [(100_000_000, 23_450), (100_000_000, 9_980), (5_000_000, 63_600)]:
    le = tien * 0.95 / gia_cp
    print(f"tiền {tien:>12,} · giá {gia_cp:>7,} → {le:>9,.1f} cp lẻ → {int(le // 100) * 100:>6,} cp lô chẵn")

tiền  100,000,000 · giá  23,450 →   4,051.2 cp lẻ →  4,000 cp lô chẵn
tiền  100,000,000 · giá   9,980 →   9,519.0 cp lẻ →  9,500 cp lô chẵn
tiền    5,000,000 · giá  63,600 →      74.7 cp lẻ →      0 cp lô chẵn


Dòng cuối: 5 triệu đồng không đủ mua một lô cổ phiếu 63.600 đồng (một lô là
6,36 triệu) — sizer trả **0** và backtrader không đặt lệnh nào. Với tài khoản nhỏ và cổ phiếu giá cao,
"mua theo tín hiệu" có thể không bao giờ xảy ra.

## 4 · Biên độ và bước giá

Giá trần/sàn = giá tham chiếu × (1 ± biên độ), rồi làm tròn về bước giá: **trần
tròn xuống, sàn tròn lên** — vì giá đặt phải nằm *trong* biên độ. HOSE có ba bậc
bước giá; HNX và UPCOM một bước 100 đồng.

In [5]:
BIEN_DO_TAY = {"HOSE": 0.07, "HNX": 0.10, "UPCOM": 0.15}


def buoc_gia_tay(gia, san="HOSE"):
    if san == "HOSE":
        return 10.0 if gia < 10_000 else 50.0 if gia < 50_000 else 100.0
    return 100.0


def tran_san_tay(tham_chieu, san="HOSE"):
    tran = tham_chieu * (1 + BIEN_DO_TAY[san])
    san_ = tham_chieu * (1 - BIEN_DO_TAY[san])
    b_tran, b_san = buoc_gia_tay(tran, san), buoc_gia_tay(san_, san)
    return math.floor(tran / b_tran + 1e-9) * b_tran, math.ceil(san_ / b_san - 1e-9) * b_san


vi_du = []
for san in ["HOSE", "HNX", "UPCOM"]:
    for tc in [9_500, 23_450, 63_600]:
        tran, san_ = tran_san_tay(tc, san)
        vi_du.append({"sàn": san, "tham chiếu": tc, "trần": tran, "sàn ": san_, "trần thật / tham chiếu": tran / tc - 1})
pd.DataFrame(vi_du).style.format({"tham chiếu": "{:,.0f}", "trần": "{:,.0f}", "sàn ": "{:,.0f}", "trần thật / tham chiếu": "{:+.2%}"})

,sàn,tham chiếu,trần,sàn,trần thật / tham chiếu
0,HOSE,"9,500","10,150","8,840",+6.84%
1,HOSE,"23,450","25,050","21,850",+6.82%
2,HOSE,"63,600","68,000","59,200",+6.92%
3,HNX,"9,500","10,400","8,600",+9.47%
4,HNX,"23,450","25,700","21,200",+9.59%
5,HNX,"63,600","69,900","57,300",+9.91%
6,UPCOM,"9,500","10,900","8,100",+14.74%
7,UPCOM,"23,450","26,900","20,000",+14.71%
8,UPCOM,"63,600","73,100","54,100",+14.94%


Cột cuối cho thấy vì sao bước giá quan trọng: trần thật của mã 9.500 đồng trên
HOSE là **+6,84%**, không phải +7% — vì 10.165 vượt ngưỡng 10.000 nên phải tròn
về bước 50 đồng; mã 9.500 đồng trên HNX chỉ được **+9,47%**. Nhận diện "phiên tăng
trần" bằng `thay_doi >= 7%` sẽ bỏ sót những phiên đó.

## 5 · Cạm bẫy: biên độ phải tính trên giá thô

Giá **điều chỉnh** nhân ngược mọi giá *trước* ngày giao dịch không hưởng quyền
với một hệ số, để lợi suất liền mạch qua cổ tức và chia tách. Đúng thứ cần cho
lợi suất — nhưng biên độ là luật về **giá thật trên bảng điện**. Áp công thức
trần/sàn lên giá điều chỉnh cho ra những mức giá chưa từng tồn tại.

Đo trên 10 mã × 5 năm: tính trần/sàn từ giá đóng cửa phiên trước, một lần bằng
giá thô, một lần bằng giá điều chỉnh.

In [6]:
def kiem_bien_do(df):
    """Trần/sàn từng phiên từ giá đóng cửa phiên trước, và ba phép phân loại phiên."""
    d = df.sort_values(["symbol", "date"]).copy()
    d["tham_chieu"] = d.groupby("symbol")["close"].shift() * 1_000
    d = d.dropna(subset=["tham_chieu"])
    d["tran"], d["san"] = zip(*[tran_san_tay(t, SAN[s]) for t, s in zip(d["tham_chieu"], d["symbol"])])
    dong_cua = d["close"] * 1_000
    d["dong_tran"] = np.isclose(dong_cua, d["tran"])
    d["dong_san"] = np.isclose(dong_cua, d["san"])
    d["ngoai_bien_do"] = (dong_cua > d["tran"] + 0.5) | (dong_cua < d["san"] - 0.5)
    return d[["symbol", "date", "tran", "san", "dong_tran", "dong_san", "ngoai_bien_do"]]


tho = kiem_bien_do(gia_tho)
dc = kiem_bien_do(gia)
ghep = tho.merge(dc, on=["symbol", "date"], suffixes=("_tho", "_dc"))

lech_tran = ghep["tran_tho"] != ghep["tran_dc"]
print(f"{len(ghep):,} phiên được so")
print(f"Trần tính từ giá điều chỉnh ≠ trần thật: {lech_tran.sum():,} phiên ({lech_tran.mean():.1%})")
print(f"  lệch trung vị ở các phiên đó: {(ghep['tran_dc'] / ghep['tran_tho'] - 1)[lech_tran].abs().median():.1%}")

bang_bd = pd.DataFrame(
    {
        "giá thô": [tho["dong_tran"].sum(), tho["dong_san"].sum(), tho["ngoai_bien_do"].sum()],
        "giá điều chỉnh": [dc["dong_tran"].sum(), dc["dong_san"].sum(), dc["ngoai_bien_do"].sum()],
    },
    index=["phiên đóng cửa đúng giá trần", "phiên đóng cửa đúng giá sàn", "phiên đóng cửa NGOÀI biên độ (không thể có)"],
)
bang_bd["sai lệch"] = bang_bd["giá điều chỉnh"] - bang_bd["giá thô"]
bang_bd

12,440 phiên được so
Trần tính từ giá điều chỉnh ≠ trần thật: 10,891 phiên (87.5%)
  lệch trung vị ở các phiên đó: 19.9%


,giá thô,giá điều chỉnh,sai lệch
phiên đóng cửa đúng giá trần,552,165,-387
phiên đóng cửa đúng giá sàn,561,174,-387
phiên đóng cửa NGOÀI biên độ (không thể có),22,422,400


In [7]:
theo_ma = (
    ghep.groupby("symbol")
    .agg(
        tran_tho=("dong_tran_tho", "sum"),
        tran_dc=("dong_tran_dc", "sum"),
        ngoai_tho=("ngoai_bien_do_tho", "sum"),
        ngoai_dc=("ngoai_bien_do_dc", "sum"),
        ty_le_tran_sai=("tran_tho", lambda s: (s != ghep.loc[s.index, "tran_dc"]).mean()),
    )
    .rename(
        columns={
            "tran_tho": "đóng trần (thô)",
            "tran_dc": "đóng trần (điều chỉnh)",
            "ngoai_tho": "ngoài biên độ (thô)",
            "ngoai_dc": "ngoài biên độ (điều chỉnh)",
            "ty_le_tran_sai": "% phiên trần sai",
        }
    )
)
theo_ma.style.format({"% phiên trần sai": "{:.0%}"})

,đóng trần (thô),đóng trần (điều chỉnh),ngoài biên độ (thô),ngoài biên độ (điều chỉnh),% phiên trần sai
symbol,,,,,
CII,69,21,1,51,78%
DIG,83,21,2,70,98%
DXG,54,9,2,52,92%
GEX,52,13,2,58,91%
HHS,51,35,3,10,56%
NVL,41,7,1,37,99%
PDR,35,12,1,35,76%
TCH,52,16,4,33,97%
VIX,66,21,4,42,97%


Ba điều bảng trên nói:

1. **Giá trần tuyệt đối gần như luôn sai** trên giá điều chỉnh: **87,5%** số
   phiên, lệch trung vị **19,9%** — mọi phiên trước lần chia cổ tức/chia tách gần
   nhất. Một lệnh "mua giá trần" đặt từ chuỗi điều chỉnh sẽ nằm ngoài biên độ và
   bị sàn từ chối.
2. **Phiên trần/sàn bị đếm thiếu.** Giá điều chỉnh không nằm trên lưới bước giá,
   nên phép so "đóng cửa = giá trần" chỉ tìm thấy **165** phiên đóng cửa trần,
   trong khi giá thô có **552**. Phiên sàn cũng vậy: 174 so với 561.
3. **Phiên "không thể có" xuất hiện hàng loạt.** Trên giá điều chỉnh, **422**
   phiên đóng cửa vượt trần hoặc thủng sàn tính theo công thức — một bộ lọc dựa
   trên chúng sẽ đánh dấu sai hoặc vứt bỏ dữ liệu thật.

Giá thô cũng có **22** phiên "ngoài biên độ" — đó là **ngày giao dịch không hưởng
quyền**: sàn điều chỉnh giá tham chiếu xuống, nên giá đóng cửa phiên trước không
còn là tham chiếu. Đó là giới hạn của cách tính tham chiếu bằng `close[-1]`,
không phải lỗi của dữ liệu.

### Vậy backtest dùng chuỗi nào?

Lợi suất cần giá **điều chỉnh**; biên độ cần giá **thô**. Bộ chặn trần/sàn ở mục
6 giải quyết bằng cách không dùng mức giá tuyệt đối: nó nhận diện phiên **trắng
bên** bằng hai điều kiện bất biến khi nhân cả chuỗi với một hệ số —

- `high == low`: cả phiên chỉ có một mức giá;
- `close / close[-1] − 1` gần chạm biên độ (dung sai 0,3% cho bước giá).

Kiểm điều đó: phân loại phiên trắng bên trên cả hai chuỗi, đếm chỗ khác nhau.

In [8]:
def trang_ben(df, dung_sai=0.003):
    d = df.sort_values(["symbol", "date"]).copy()
    thay_doi = d["close"] / d.groupby("symbol")["close"].shift() - 1
    nguong = d["symbol"].map(SAN).map(BIEN_DO_TAY) - dung_sai
    mot_gia = d["high"] == d["low"]
    d["tran_trang"] = mot_gia & (thay_doi >= nguong)
    d["san_trang"] = mot_gia & (thay_doi <= -nguong)
    return d[["symbol", "date", "tran_trang", "san_trang"]]


tb = trang_ben(gia_tho).merge(trang_ben(gia), on=["symbol", "date"], suffixes=("_tho", "_dc"))
for c in ["tran_trang", "san_trang"]:
    print(f"{c}: thô {tb[c + '_tho'].sum()} phiên · điều chỉnh {tb[c + '_dc'].sum()} phiên · khác nhau {(tb[c + '_tho'] != tb[c + '_dc']).sum()} phiên")

tran_trang: thô 12 phiên · điều chỉnh 14 phiên · khác nhau 2 phiên
san_trang: thô 45 phiên · điều chỉnh 43 phiên · khác nhau 2 phiên


Phép nhận diện theo tỷ lệ cho kết quả gần như trùng nhau trên hai chuỗi — chỉ
khác 2 phiên mỗi loại trên hơn 12 nghìn phiên — nên nó chạy được thẳng trên feed
giá điều chỉnh. Phần khác biệt nhỏ còn lại rơi vào
ngày không hưởng quyền, và ở đó chuỗi điều chỉnh mới là chuỗi đúng: tham chiếu
của sàn chính là giá đã điều chỉnh.

## 6 · Bộ chặn trần/sàn trắng bên: `filler`

`cerebro.broker.set_filler(f)` cho phép quyết định **khớp được bao nhiêu** mỗi
khi broker định khớp một lệnh. Trả về 0 thì lệnh không khớp và chờ phiên sau.

Giả định ở đây là bảo thủ: phiên trần mà `high == low` thì có người mua được —
nhưng là người xếp hàng từ trước, không phải một lệnh thị trường đặt từ tối hôm
trước.

In [9]:
class ChanTay:
    """Không khớp lệnh mua ở phiên trần trắng bên, lệnh bán ở phiên sàn trắng bên."""

    def __init__(self, san="HOSE", dung_sai=0.003):
        self.bien_do = BIEN_DO_TAY[san]
        self.dung_sai = dung_sai
        self.chan_mua = 0
        self.chan_ban = 0

    def __call__(self, order, price, ago):
        d = order.data
        con_lai = abs(order.executed.remsize)
        if len(d) < 2 or d.high[ago] != d.low[ago]:
            return con_lai
        thay_doi = d.close[ago] / d.close[ago - 1] - 1
        nguong = self.bien_do - self.dung_sai
        if order.isbuy() and thay_doi >= nguong:
            self.chan_mua += 1
            return 0
        if order.issell() and thay_doi <= -nguong:
            self.chan_ban += 1
            return 0
        return con_lai

## 7 · T+2 và không bán khống

Hai luật cuối cùng nằm trong **strategy**, vì chúng phụ thuộc vào lịch sử lệnh:

- **T+2**: cổ phiếu mua (khớp) ở phiên T bán được từ phiên T+2. Đếm theo *phiên
  giao dịch có trong feed*, không theo ngày lịch — nghỉ lễ không làm hàng về sớm.
- **Không bán khống**: chỉ bán số cổ phiếu đã về tài khoản.

Trước hết, xem backtrader mặc định làm gì khi bán thứ mình không có:

In [10]:
class BanKhong(bt.Strategy):
    def __init__(self):
        self.vi_the = []

    def next(self):
        if len(self) == 10:
            self.sell(size=1_000)  # chưa từng mua
        if len(self) in (10, 11, 12):
            self.vi_the.append((len(self), self.position.size, self.broker.getcash()))


cerebro = bt.Cerebro(stdstats=False)
cerebro.adddata(bt_vn.feed_tu_finlens(gia, ma=RO[0]))
cerebro.broker.setcash(100_000_000)
cerebro.addstrategy(BanKhong)
for n, sl, tien in cerebro.run()[0].vi_the:
    print(f"phiên {n}: vị thế {sl:+,} cp · tiền mặt {tien:,.0f}")

phiên 10: vị thế +0 cp · tiền mặt 100,000,000
phiên 11: vị thế -1,000 cp · tiền mặt 118,740,000
phiên 12: vị thế -1,000 cp · tiền mặt 118,740,000


Vị thế âm 1.000 cổ phiếu, tiền mặt **tăng** thêm tiền bán — không lỗi, không cảnh
báo. Với cổ phiếu Việt Nam điều đó không thể xảy ra. Lớp nền dưới đây chặn cả
hai: sổ ghi từng lô mua theo phiên khớp, và `ban()` chỉ bán phần đã đủ T+2 **và
chưa nằm trong một lệnh bán đang chờ**. Vế sau quan trọng khi có bộ chặn sàn ở
mục 6: lệnh bán bị chặn không bị huỷ, nên nếu chiến lược đặt bán thêm lần nữa thì
hai lệnh cùng khớp ở phiên mở cửa sau — và vị thế thành âm.

In [11]:
class ChienLuocTay(bt.Strategy):
    """Lớp nền: sổ T+2 và lệnh bán không vượt quá số đã về."""

    params = (("t_cong", 2),)

    def __init__(self):
        self.so_lo = deque()  # (phiên khớp, số cp) — mua trước bán trước

    def notify_order(self, order):
        if order.status != order.Completed:
            return
        if order.isbuy():
            self.so_lo.append((len(self.data), order.executed.size))
        else:
            can_tru = -order.executed.size
            while can_tru > 0 and self.so_lo:
                phien, sl = self.so_lo[0]
                if sl <= can_tru:
                    can_tru -= sl
                    self.so_lo.popleft()
                else:
                    self.so_lo[0] = (phien, sl - can_tru)
                    can_tru = 0

    def ban_duoc(self):
        phien_khop = len(self.data) + 1  # lệnh đặt hôm nay khớp ở phiên sau
        da_ve = sum(sl for p, sl in self.so_lo if phien_khop - p >= self.p.t_cong)
        # Lệnh bán bị chặn ở phiên sàn vẫn còn sống — không trừ ra thì đặt bán lần hai là bán khống
        dang_cho = sum(abs(o.executed.remsize) for o in self.broker.get_orders_open() if o.issell())
        return int(max(0, da_ve - dang_cho))

    def ban(self):
        sl = self.ban_duoc()
        return self.sell(size=sl) if sl > 0 else None


class BanKhongVN(ChienLuocTay):
    def next(self):
        if len(self) == 10:
            print(f"phiên 10: ban() trả về {self.ban()} · vị thế {self.position.size}")
        if len(self) == 20:
            self.buy(size=1_000)
        if 21 <= len(self) <= 23:
            print(f"phiên {len(self)}: giữ {self.position.size:,} cp · ban_duoc() = {self.ban_duoc():,}")


cerebro = bt.Cerebro(stdstats=False)
cerebro.adddata(bt_vn.feed_tu_finlens(gia, ma=RO[0]))
cerebro.broker.setcash(100_000_000)
cerebro.addstrategy(BanKhongVN)
_ = cerebro.run()

phiên 10: ban() trả về None · vị thế 0


phiên 21: giữ 1,000 cp · ban_duoc() = 0
phiên 22: giữ 1,000 cp · ban_duoc() = 1,000
phiên 23: giữ 1,000 cp · ban_duoc() = 1,000


Lệnh mua đặt ở phiên 20, khớp ở phiên 21 — đó là ngày T. `ban_duoc()` hỏi "nếu
đặt lệnh bán **bây giờ** thì lúc khớp (phiên sau) hàng đã về chưa":

- ở phiên 21, lệnh bán sẽ khớp phiên 22 = T+1 → chưa về, trả **0**;
- ở phiên 22, lệnh bán sẽ khớp phiên 23 = T+2 → về rồi, trả **1.000**.

Ghi chú: thực tế hàng về khoảng giữa phiên T+2, nên khớp ở **giá mở cửa** T+2
là hơi lạc quan. Còn tiền bán được dùng để mua ngay trong phiên (ứng trước tiền
bán là dịch vụ phổ biến), nên notebook không chặn chiều tiền.

## 8 · Cùng một chiến lược, sáu bộ luật

Chiến lược **bứt phá**: mua khi giá đóng cửa vượt đỉnh 20 phiên trước; bán khi
đóng cửa thủng đáy 10 phiên trước, hoặc lỗ 5% so với giá mua. Loại chiến lược
này mua đúng những phiên tăng mạnh và cắt lỗ nhanh — tức va vào trần/sàn và T+2
nhiều nhất.

Logic viết **một lần**, trên một lớp nền truyền vào — để chạy được với cả lớp
nền tay ở trên và `bt_vn.ChienLuocVN` ở cuối bài.

In [12]:
def tao_bu_pha(nen):
    class BuPha(nen):
        params = (("dinh", 20), ("day", 10), ("cat_lo", 0.05), ("t2", True))

        def __init__(self):
            super().__init__()
            self.dinh = bt.ind.Highest(self.data.close(-1), period=self.p.dinh)
            self.day = bt.ind.Lowest(self.data.close(-1), period=self.p.day)
            self.lenh_cho = None
            self.muon_ban = False
            self.gia_vao = None
            self.lan_hoan_t2 = 0  # số lần muốn bán nhưng hàng chưa về
            self.phien_cho_t2 = 0  # tổng số phiên phải chờ

        def notify_order(self, order):
            super().notify_order(order)
            if order.status == order.Completed and order.isbuy():
                self.gia_vao = order.executed.price
            if not order.alive():
                self.lenh_cho = None

        def next(self):
            if self.lenh_cho:
                return
            c = self.data.close[0]
            if not self.position:
                if c > self.dinh[0]:
                    self.lenh_cho = self.buy()  # khối lượng do sizer quyết
                return
            if not self.muon_ban and (c < self.day[0] or c < self.gia_vao * (1 - self.p.cat_lo)):
                self.muon_ban = True
                if self.p.t2 and self.ban_duoc() <= 0:
                    self.lan_hoan_t2 += 1
            if self.muon_ban:
                lenh = self.ban() if self.p.t2 else self.close()
                if lenh is None:
                    self.phien_cho_t2 += 1
                else:
                    self.lenh_cho, self.muon_ban = lenh, False

    return BuPha


BuPhaTay = tao_bu_pha(ChienLuocTay)

Sáu bộ luật, mỗi bộ thêm đúng **một** luật vào bộ trước — để thấy từng luật
đóng góp bao nhiêu vào khoảng cách cuối cùng.

In [13]:
VON = 100_000_000
BAC = [
    ("0 · ngây thơ", dict(phi=False, thue=False, lo=False, chan=False, t2=False)),
    ("1 · + phí 0,15% hai chiều", dict(phi=True, thue=False, lo=False, chan=False, t2=False)),
    ("2 · + thuế bán 0,1%", dict(phi=True, thue=True, lo=False, chan=False, t2=False)),
    ("3 · + lô 100", dict(phi=True, thue=True, lo=True, chan=False, t2=False)),
    ("4 · + chặn trần/sàn trắng bên", dict(phi=True, thue=True, lo=True, chan=True, t2=False)),
    ("5 · + T+2 = luật VN", dict(phi=True, thue=True, lo=True, chan=True, t2=True)),
]


def chay(ma, phi, thue, lo, chan, t2, *, ban_module=False):
    cerebro = bt.Cerebro(stdstats=False)
    cerebro.adddata(bt_vn.feed_tu_finlens(gia, ma=ma))
    cerebro.broker.setcash(VON)
    if ban_module:  # mọi mảnh lấy từ bt_vn
        cerebro.broker.addcommissioninfo(bt_vn.PhiCoPhieuVN())
        cerebro.addsizer(bt_vn.LoVN)
        bo_chan = bt_vn.KhopTheoBienDo(SAN[ma])
        cerebro.broker.set_filler(bo_chan)
        cerebro.addstrategy(tao_bu_pha(bt_vn.ChienLuocVN), t2=True)
    else:
        cerebro.broker.addcommissioninfo(PhiTay(phi=0.0015 if phi else 0.0, thue_ban=0.001 if thue else 0.0))
        if lo:
            cerebro.addsizer(LoTay)
        else:
            cerebro.addsizer(bt.sizers.PercentSizerInt, percents=95)
        bo_chan = ChanTay(SAN[ma]) if chan else None
        if bo_chan:
            cerebro.broker.set_filler(bo_chan)
        cerebro.addstrategy(BuPhaTay, t2=t2)
    bt_vn.them_analyzer(cerebro)
    s = cerebro.run()[0]
    kq = bt_vn.ket_qua(s)
    tt = kq["tom_tat"]
    if ban_module:
        chan_mua, chan_ban = bo_chan.so_lan_chan_mua, bo_chan.so_lan_chan_ban
    else:
        chan_mua = bo_chan.chan_mua if bo_chan else 0
        chan_ban = bo_chan.chan_ban if bo_chan else 0
    return {
        "mã": ma,
        "lệnh khớp": tt["so_lenh_khop"],
        "vòng": tt["so_vong_dong"],
        "CAGR": tt["cagr"],
        "MDD": tt["mdd"],
        "cuối kỳ": cerebro.broker.getvalue(),
        "chặn mua (trần)": chan_mua,
        "chặn bán (sàn)": chan_ban,
        "chặn (tổng)": chan_mua + chan_ban,
        "hoãn T+2": s.lan_hoan_t2,
        "phiên chờ T+2": s.phien_cho_t2,
        "giao_dich": kq["giao_dich"],
    }


ket = {ten: pd.DataFrame([chay(ma, **luat) for ma in RO]).set_index("mã") for ten, luat in BAC}
print("Chạy xong", len(BAC), "bộ luật ×", len(RO), "mã")

Chạy xong 6 bộ luật × 10 mã


`bt_vn.ket_qua` đọc analyzer ra pandas (CAGR, MDD, số lệnh) — notebook `84` sẽ
mở nó ra. Ở đây chỉ cần con số.

In [14]:
tong = pd.DataFrame(
    {
        ten: {
            "lệnh khớp": k["lệnh khớp"].sum(),
            "vòng mua-bán": k["vòng"].sum(),
            "CAGR trung bình": k["CAGR"].mean(),
            "MDD trung bình": k["MDD"].mean(),
            "lần chặn mua (trần)": k["chặn mua (trần)"].sum(),
            "lần chặn bán (sàn)": k["chặn bán (sàn)"].sum(),
            "lần hoãn bán vì T+2": k["hoãn T+2"].sum(),
        }
        for ten, k in ket.items()
    }
).T
tong.style.format(
    {
        "lệnh khớp": "{:,.0f}",
        "vòng mua-bán": "{:,.0f}",
        "CAGR trung bình": "{:+.2%}",
        "MDD trung bình": "{:.1%}",
        "lần chặn mua (trần)": "{:,.0f}",
        "lần chặn bán (sàn)": "{:,.0f}",
        "lần hoãn bán vì T+2": "{:,.0f}",
    }
)

,lệnh khớp,vòng mua-bán,CAGR trung bình,MDD trung bình,lần chặn mua (trần),lần chặn bán (sàn),lần hoãn bán vì T+2
0 · ngây thơ,517,258,+7.12%,-37.1%,0,0,0
"1 · + phí 0,15% hai chiều",517,258,+5.52%,-39.0%,0,0,0
"2 · + thuế bán 0,1%",517,258,+5.00%,-39.6%,0,0,0
3 · + lô 100,521,260,+4.80%,-39.4%,0,0,0
4 · + chặn trần/sàn trắng bên,521,260,+4.11%,-41.0%,0,5,0
5 · + T+2 = luật VN,519,259,+3.44%,-41.7%,0,5,4


In [15]:
buoc = tong["CAGR trung bình"].diff().dropna() * 100
bar_ngang(
    pd.DataFrame({"luat": buoc.index.str.split(" · ").str[1], "doi": buoc.values}),
    nhan="luat",
    gia_tri="doi",
    tieu_de="Mỗi luật Việt Nam làm CAGR trung bình đổi bao nhiêu",
    phu_de=f"Chiến lược bứt phá 20/10 phiên, cắt lỗ 5% · {len(RO)} mã HOSE · {SO_NAM} năm",
    nhan_x="điểm % mỗi năm",
    dinh_dang_nhan="{:+.2f}",
)

## 9 · Ngây thơ vs luật VN, từng mã

In [16]:
ngay_tho, luat_vn = ket[BAC[0][0]], ket[BAC[-1][0]]
so_sanh = pd.DataFrame(
    {
        "lệnh (ngây thơ)": ngay_tho["lệnh khớp"],
        "lệnh (VN)": luat_vn["lệnh khớp"],
        "CAGR (ngây thơ)": ngay_tho["CAGR"],
        "CAGR (VN)": luat_vn["CAGR"],
        "chênh CAGR (đpt)": (luat_vn["CAGR"] - ngay_tho["CAGR"]) * 100,
        "MDD (ngây thơ)": ngay_tho["MDD"],
        "MDD (VN)": luat_vn["MDD"],
        "chặn trần/sàn": luat_vn["chặn (tổng)"],
        "hoãn T+2": luat_vn["hoãn T+2"],
    }
)
print(f"Chênh CAGR trung bình: {so_sanh['chênh CAGR (đpt)'].mean():+.2f} điểm %/năm · "
      f"{(so_sanh['chênh CAGR (đpt)'] < 0).sum()}/{len(RO)} mã xấu đi khi áp luật VN")
so_sanh.style.format(
    {
        "CAGR (ngây thơ)": "{:+.2%}",
        "CAGR (VN)": "{:+.2%}",
        "chênh CAGR (đpt)": "{:+.2f}",
        "MDD (ngây thơ)": "{:.1%}",
        "MDD (VN)": "{:.1%}",
        "chặn trần/sàn": "{:,.0f}",
    }
)

Chênh CAGR trung bình: -3.68 điểm %/năm · 10/10 mã xấu đi khi áp luật VN


,lệnh (ngây thơ),lệnh (VN),CAGR (ngây thơ),CAGR (VN),chênh CAGR (đpt),MDD (ngây thơ),MDD (VN),chặn trần/sàn,hoãn T+2
mã,,,,,,,,,
CII,57,57,+3.91%,-3.51%,-7.42,-49.5%,-62.4%,3,1
DIG,50,52,+1.34%,-7.53%,-8.87,-42.0%,-49.4%,2,1
DXG,58,58,+4.74%,+2.40%,-2.34,-31.2%,-34.2%,0,0
GEX,50,50,+12.12%,+9.43%,-2.69,-31.3%,-35.0%,0,1
HHS,48,48,+13.84%,+8.69%,-5.15,-39.7%,-47.2%,0,1
NVL,48,48,+6.21%,+4.23%,-1.98,-28.9%,-30.9%,0,0
PDR,52,52,-3.67%,-5.36%,-1.69,-43.5%,-47.0%,0,0
TCH,52,52,-2.52%,-4.47%,-1.95,-39.5%,-43.0%,0,0
VIX,52,52,+31.99%,+29.34%,-2.65,-34.8%,-36.1%,0,0


Đọc hai bảng cùng nhau:

- **Cả 10/10 mã xấu đi.** CAGR trung bình giảm **3,68 điểm %/năm** (7,12% →
  3,44%) — hơn một nửa lợi nhuận của bản ngây thơ đến từ những luật bị bỏ qua.
  MDD trung bình sâu thêm từ −37,1% xuống −41,7%.
- **Phí là khoản lớn nhất** (−1,60 điểm), rồi tới **thuế bán** (−0,52). Kiểm
  nhanh: 258 vòng ÷ 10 mã ÷ 5 năm ≈ 5,2 vòng mỗi năm; × 0,3% phí ≈ 1,55%, × 0,1%
  thuế ≈ 0,52%. `setcommission(0.0015)` bắt được phần đầu, bỏ sót phần sau.
- **Lô 100 gần như vô hại** với vốn 100 triệu (−0,20) — phần lẻ dư ra chỉ là
  tiền mặt không đầu tư. Vốn nhỏ hoặc cổ phiếu giá cao thì khác (mục 3).
- **Trần/sàn trắng bên: hiếm nhưng đắt.** Chỉ 5 lần chặn trong 5 năm, đều là
  lệnh **bán** ở phiên sàn trắng bên (CII 3, DIG 2), không lệnh mua nào bị chặn
  — vậy mà kéo CAGR trung bình đi 0,69 điểm. Mỗi lần bị chặn là một lần cắt lỗ
  trễ ít nhất một phiên giữa chuỗi giảm sàn, và hai mã đó cũng là hai mã xấu đi
  nhiều nhất bảng.
- **T+2: 4 lần cắt lỗ phải hoãn**, −0,67 điểm. Chiến lược cắt lỗ càng nhanh thì
  càng hay va vào T+2.
- **Số lệnh gần như không đổi** (517 → 519). Luật Việt Nam không làm chiến lược
  giao dịch ít đi — nó làm *từng giao dịch* tệ đi, nên không lộ ra ở các thống kê
  đếm.

Và đây vẫn là kết quả **trước** trượt giá, trước giới hạn tỷ trọng khối lượng,
và trên một rổ mã chọn bằng hiểu biết của hôm nay.

## 10 · Tất cả đã có trong `bt_vn`

Mọi mảnh vừa dựng tay đã được gom vào
[`finlens_examples/bt_vn.py`](../../finlens_examples/bt_vn.py):

| Dựng tay ở đây | Trong `bt_vn` |
|---|---|
| `PhiTay` | `PhiCoPhieuVN(phi=0.0015, thue_ban=0.001)` |
| `LoTay` | `LoVN(ty_le=0.95, lo=100)` |
| `buoc_gia_tay`, `tran_san_tay` | `buoc_gia(gia, san)`, `tran_san(tham_chieu, san)` |
| `ChanTay` (`chan_mua`, `chan_ban`) | `KhopTheoBienDo(san)` (`so_lan_chan_mua`, `so_lan_chan_ban`; tổng `so_lan_chan`) |
| `ChienLuocTay` | `ChienLuocVN` — `ban_duoc()`, `ban()`; `LoVN` chiều bán cũng chỉ bán phần `ban_duoc()` |

Kiểm từng mảnh cho cùng kết quả. Trước hết hai hàm giá, trên cả một lưới giá
tham chiếu của ba sàn:

In [17]:
luoi = np.arange(1_000, 200_001, 10, dtype=float)
for san in ["HOSE", "HNX", "UPCOM"]:
    lech_buoc = sum(buoc_gia_tay(g, san) != bt_vn.buoc_gia(g, san) for g in luoi)
    lech_ts = sum(tran_san_tay(g, san) != bt_vn.tran_san(g, san) for g in luoi)
    print(f"{san:<6} {len(luoi):,} mức giá · lệch bước giá: {lech_buoc} · lệch trần/sàn: {lech_ts}")

print(f"\nPhí: tay {PhiTay().getcommission(-1_000, 21_000):,.0f} · module {bt_vn.PhiCoPhieuVN().getcommission(-1_000, 21_000):,.0f} VND")

HOSE   19,901 mức giá · lệch bước giá: 0 · lệch trần/sàn: 0
HNX    19,901 mức giá · lệch bước giá: 0 · lệch trần/sàn: 0
UPCOM  19,901 mức giá · lệch bước giá: 0 · lệch trần/sàn: 0

Phí: tay 52,500 · module 52,500 VND


Rồi cả chiến lược: chạy bộ luật VN đầy đủ bằng **toàn bộ mảnh của module**
(`PhiCoPhieuVN`, `LoVN`, `KhopTheoBienDo`, `ChienLuocVN`), so với bản tay từng
mã — số lệnh, số lần chặn mua và chặn bán, số lần hoãn T+2, giá trị cuối kỳ, và
**từng giao dịch**.

In [18]:
doi_chieu = []
for ma in RO:
    m = chay(ma, **BAC[-1][1], ban_module=True)
    t = ket[BAC[-1][0]].loc[ma]
    cung_giao_dich = m["giao_dich"].reset_index(drop=True).equals(t["giao_dich"].reset_index(drop=True))
    doi_chieu.append(
        {
            "mã": ma,
            "lệnh (tay)": t["lệnh khớp"],
            "lệnh (module)": m["lệnh khớp"],
            "chặn mua (tay)": t["chặn mua (trần)"],
            "chặn mua (module)": m["chặn mua (trần)"],
            "chặn bán (tay)": t["chặn bán (sàn)"],
            "chặn bán (module)": m["chặn bán (sàn)"],
            "hoãn T+2 (tay)": t["hoãn T+2"],
            "hoãn T+2 (module)": m["hoãn T+2"],
            "lệch cuối kỳ (VND)": m["cuối kỳ"] - t["cuối kỳ"],
            "giao dịch trùng": cung_giao_dich,
        }
    )
doi_chieu = pd.DataFrame(doi_chieu).set_index("mã")
print(f"{doi_chieu['giao dịch trùng'].sum()}/{len(RO)} mã trùng từng giao dịch · lệch cuối kỳ lớn nhất {doi_chieu['lệch cuối kỳ (VND)'].abs().max():,.2f} VND")
doi_chieu

10/10 mã trùng từng giao dịch · lệch cuối kỳ lớn nhất 0.00 VND


,lệnh (tay),lệnh (module),chặn mua (tay),chặn mua (module),chặn bán (tay),chặn bán (module),hoãn T+2 (tay),hoãn T+2 (module),lệch cuối kỳ (VND),giao dịch trùng
mã,,,,,,,,,,
CII,57,57,0,0,3,3,1,1,0.0,True
DIG,52,52,0,0,2,2,1,1,0.0,True
DXG,58,58,0,0,0,0,0,0,0.0,True
GEX,50,50,0,0,0,0,1,1,0.0,True
HHS,48,48,0,0,0,0,1,1,0.0,True
NVL,48,48,0,0,0,0,0,0,0.0,True
PDR,52,52,0,0,0,0,0,0,0.0,True
TCH,52,52,0,0,0,0,0,0,0.0,True
VIX,52,52,0,0,0,0,0,0,0.0,True


Từ notebook `84` trở đi, mọi chiến lược dùng thẳng các mảnh này:

```python
cerebro.broker.addcommissioninfo(bt_vn.PhiCoPhieuVN())
cerebro.addsizer(bt_vn.LoVN)
cerebro.broker.set_filler(bt_vn.KhopTheoBienDo("HOSE"))

class ChienLuocCuaBan(bt_vn.ChienLuocVN):
    def __init__(self):
        super().__init__()          # bắt buộc — không gọi thì sổ T+2 rỗng
        ...
    def notify_order(self, order):
        super().notify_order(order)  # bắt buộc, cùng lý do
        ...
    def next(self):
        ...
        self.ban()                   # thay cho self.close()
```

## Tổng kết

| Luật | Dựng bằng | Cạm bẫy |
|---|---|---|
| Phí + thuế bán | `CommInfoBase._getcommission`, `size < 0` là bán | `setcommission` tính một tỷ lệ hai chiều — thiếu thuế |
| Lô 100 | `Sizer._getsizing`, làm tròn xuống | tài khoản nhỏ + giá cao → sizer trả 0, không lệnh nào |
| Biên độ, bước giá | `tran_san`, `buoc_gia` trên **giá thô** | giá điều chỉnh cho trần/sàn không tồn tại |
| Trần/sàn trắng bên | `broker.set_filler`, nhận diện theo **tỷ lệ** + `high == low` | chạy được trên giá điều chỉnh vì không dùng mức giá tuyệt đối |
| T+2 | sổ lô theo **phiên khớp** trong strategy | `self.close()` mặc định bán ngay phiên sau |
| Không bán khống | `ban()` chỉ bán phần đã về | `self.sell()` không có hàng → vị thế âm, không lỗi |

---

**Bậc 1 của Track 8 xong.** Tiếp theo: `84_analyzer_va_bao_cao.ipynb` —
analyzer → pandas → plotly, VNINDEX làm chuẩn, lãi suất phi rủi ro từ `client.macro`.